# dmpbridge — Experiment Log

Records current model experiments, prompting strategies, and results.

**Dataset:** 10 manually labeled DMP documents, 741 blocks total  
**Evaluation:** Block-level label accuracy and per-label F1 (see per-model eval notebooks in `notebooks/`)  
**Labels:** `title`, `section.title`, `section.description`, `question.text`, `answer.text`  
**Output directory:** `data/output/labeled/`

## Experiment Registry

| ID | Model | Strategy | Output tag | Samples | Accuracy | Status |
|---|---|---|---|---|---|---|
| M01 | Claude Opus 4.8 | batch | `claude-opus-4-8_batch` | 10/10 | 94.9% | Done |
| M02 | Claude Opus 4.8 | whole-doc | `claude-opus-4-8_whole_doc` | 10/10 | 96.9% | Done |
| M03 | Llama 3.3 70B | batch | `llama3.3-70b_batch` | 10/10 | 94.1% | Done |
| M04 | Llama 3.3 70B | whole-doc | `llama3.3-70b_whole_doc` | 10/10 | 91.8% | Done |
| M05 | Llama 3.1 8B | batch | `llama3.1-8b_batch` | 10/10 | 67.3% | Done |
| M06 | Llama 3.1 8B | whole-doc | `llama3.1-8b_whole_doc` | 10/10 | 84.2% | Done |
| M07 | Claude Opus 4.8 | pdf-direct | `claude-opus-4-8_pdf` | 10/10 | 95.1% (99.5%*) | Done |
| F01 | Llama 3.1 8B fine-tune | — | — | — | — | Planned |

Output files follow the pattern: `data/output/labeled/{output-tag}/sampleN.json`  
\* PDF-direct accuracy: 95.1% block-level (193/203 blocks), 99.5% gold-based (correct/741 gold items).  
Detailed per-label analysis: `notebooks/003_prompt_strategy_comparison.ipynb` and per-model eval notebooks.

---
## Prompting Strategies

### Batch

The document is classified in overlapping windows of 10 blocks. Each window carries 3 blocks of context from the previous window to preserve label continuity across boundaries. The model makes one API call per window.

- Context window: 10 blocks, 3-block overlap
- API calls per document: approximately total blocks / 7
- Provider: Ollama (local) or Anthropic API
- Inference: `dmpbridge-experiment experiments/{name}-batch.yaml`

### Whole-document

All extracted blocks from the document are passed to the model in a single API call. The model classifies the entire document at once, with full visibility into structure and context.

- Context window: full document (typically 60–100 blocks)
- API calls per document: 1
- Provider: Ollama (local) or Anthropic API
- Inference: `dmpbridge-wholedoc` CLI

### PDF-direct

The raw PDF is sent directly to a vision-capable model (Claude). No pdfplumber extraction — Claude reads the PDF and classifies paragraph-level blocks in a single call.

- Context window: full PDF (raw bytes)
- API calls per document: 1
- Provider: Anthropic API only (requires vision capability)
- Block granularity: paragraph-level (~20 blocks per document vs ~74 for pdfplumber)
- Inference: `dmpbridge-pdf --model claude-opus-4-8`

---
## Claude Opus 4.8

**Provider:** Anthropic API  
**Model ID:** `claude-opus-4-8`  
**Input:** `data/input/pdfs/sampleN.pdf` (10 files)  
**Evaluation notebook:** `notebooks/006_claude-opus-4-8.ipynb`

### M01 — Batch

**Output files:** `data/output/labeled/claude-opus-4-8_batch/sampleN.json`  
**Inference:** `dmpbridge` CLI with `--provider anthropic --model claude-opus-4-8`

| Metric | Value |
|---|---|
| Overall accuracy | 94.9% (703/741) |
| Samples complete | 10/10 |

Strong baseline across all label types. Detailed per-label F1 in eval notebook.

### M02 — Whole-document

**Output files:** `data/output/labeled/claude-opus-4-8_whole_doc/sampleN.json`  
**Inference:** `dmpbridge-wholedoc --provider anthropic --model claude-opus-4-8`  
**Note:** Uses `max_tokens=16384` to accommodate full structured output.

| Metric | Value |
|---|---|
| Overall accuracy | 96.9% (718/741) |
| Samples complete | 10/10 |
| Delta vs batch | +2.0pp |

Full-document context improves `question.text` F1 by +26pp and `section.description` F1 by +15pp. The model handles structural ambiguity better when it can see surrounding sections. Whole-document is the recommended strategy for Claude Opus 4.8.

### M07 — PDF-direct

**Output files:** `data/output/labeled/claude-opus-4-8_pdf/sampleN.json`  
**Inference:** `dmpbridge-pdf --model claude-opus-4-8`  
**Note:** Sends raw PDF bytes to the Claude vision API — no pdfplumber extraction.

| Metric | Value |
|---|---|
| Block accuracy | 95.1% (193/203 paragraph blocks) |
| Gold accuracy | 99.5% (correct / 741 gold items) |
| Samples complete | 10/10 |

Highest gold accuracy of all strategies. Claude extracts and classifies paragraph-level blocks from the raw PDF, covering more gold items per block. The lower block count (203 vs 741) means each block covers more text, reducing fragmentation errors. Does not produce bounding-box coordinates.

---
## Llama 3.3 70B

**Provider:** Ollama (local)  
**Model ID:** `llama3.3:70b` (Q4_K_M, ~42 GB)  
**Input:** `data/input/pdfs/sampleN.pdf` (10 files)  
**Evaluation notebook:** `notebooks/005_llama3.3-70b.ipynb`

### M03 — Batch

**Output files:** `data/output/labeled/llama3.3-70b_batch/sampleN.json`  
**Inference:** `dmpbridge` CLI with `--provider ollama --model llama3.3:70b`

| Metric | Value |
|---|---|
| Overall accuracy | 94.1% (697/741) |
| Samples complete | 10/10 |

Competitive accuracy with Claude Opus 4.8 batch (−0.8pp). Runs fully offline. Detailed per-label F1 in eval notebook.

### M04 — Whole-document

**Output files:** `data/output/labeled/llama3.3-70b_whole_doc/sampleN.json`  
**Inference:** `dmpbridge-wholedoc --provider ollama --model llama3.3:70b`

| Metric | Value |
|---|---|
| Overall accuracy | 91.8% (680/741) |
| Samples complete | 10/10 |
| Delta vs batch | −2.3pp |

Whole-document prompting hurts for this model. With full document context, the model over-classifies `question.text` blocks as `section.description`, producing a large drop in `question.text` F1 (−36pp). The batch strategy is the recommended approach for Llama 3.3 70B.

---
## Llama 3.1 8B

**Provider:** Ollama (local)  
**Model ID:** `llama3.1:8b` (Q4_K_M, ~5 GB)  
**Input:** `data/input/pdfs/sampleN.pdf` (10 files)  
**Evaluation notebook:** `notebooks/004_llama3.1-8b.ipynb`

### M05 — Batch

**Output files:** `data/output/labeled/llama3.1-8b_batch/sampleN.json`  
**Inference:** `dmpbridge` CLI with `--provider ollama --model llama3.1:8b`

| Metric | Value |
|---|---|
| Overall accuracy | 67.3% (499/741) |
| Samples complete | 10/10 |

Substantially below the 70B model. The 8B model cannot reliably distinguish `section.description` (funder-written) from `question.text` (researcher-written) in batch mode — few-shot examples are not sufficient to bridge this gap at this model size.

### M06 — Whole-document

**Output files:** `data/output/labeled/llama3.1-8b_whole_doc/sampleN.json`  
**Inference:** `dmpbridge-wholedoc --provider ollama --model llama3.1:8b`

| Metric | Value |
|---|---|
| Overall accuracy | 84.2% (624/741) |
| Samples complete | 10/10 |
| Delta vs batch | +16.9pp |

Whole-document context produces a large accuracy gain for the 8B model (+16.9pp overall, +22pp `question.text` F1). Full document visibility compensates for the model's weaker semantic discrimination in batch mode. Even so, 84.2% is still 10pp below Llama 3.3 70B batch and 13pp below Claude whole-doc, and is not suitable for production use without fine-tuning.

---
## Cross-model Summary

| Model | Batch | Whole-doc | PDF-direct\* | Best | Recommendation |
|---|---|---|---|---|---|
| Claude Opus 4.8 | 94.9% | 96.9% | **99.5%** | PDF-direct | Use whole-doc (or pdf-direct for highest coverage) |
| Llama 3.3 70B | 94.1% | 91.8% | — | Batch | Use batch |
| Llama 3.1 8B | 67.3% | 84.2% | — | Whole-doc | Neither — see F01 |

\* PDF-direct accuracy is gold-based (correct / 741 gold items); block accuracy is 95.1% (193/203 paragraph blocks).

**Key findings:**

- Whole-document context helps Claude and small models, but hurts Llama 3.3 70B. The 70B model becomes over-confident about document-level structure and misclassifies individual block roles.
- Llama 3.3 70B batch is the best offline option at 94.1%, close to Claude batch (94.9%) at no API cost.
- Llama 3.1 8B whole-doc at 84.2% may be acceptable for low-resource deployments where accuracy requirements are relaxed, but is not recommended for production use.
- The accuracy gap between the best Llama configuration (94.1%) and Claude whole-doc (96.9%) is 2.8pp across 741 blocks — approximately 21 additional correct labels per 741.
- Claude PDF-direct achieves the highest gold accuracy (99.5%) by sending the raw PDF directly to the model — no pdfplumber extraction required, but bounding-box coordinates are not available and the Anthropic API is required.

Detailed per-label F1 and per-sample breakdowns: `notebooks/003_prompt_strategy_comparison.ipynb`

---
## Future Plans

### F01 — Fine-tune Llama 3.1 8B

**Status:** Planned  
**Depends on:** Expanding the labeled dataset to at least 20 samples (need held-out evaluation set)

**Hypothesis:** Fine-tuning on labeled DMP blocks will close most of the accuracy gap between Llama 3.1 8B and Llama 3.3 70B, making low-resource deployment viable.

**Planned approach:**
- Generate block-level training pairs from `data/input/ground_truth/`
- Fine-tune with LoRA or QLoRA
- Evaluate on a held-out set not used in training
- Target: `question.text` F1 from 33% (batch) to 70%+

**Blocker:** The current 10-sample dataset is too small to split into train/eval. Dataset expansion is a prerequisite.

---

*Other possible future work: sequence correction post-processing (state machine over label sequence); dataset expansion to non-NIH/NSF funders.*